### **1\. Proposition: Top 3 Salespersons per Territory by Total Sales**

<span style="color: var(--vscode-foreground);">Sales managers need to identify top-performing salespeople in each territory to guide recognition, bonuses, and training needs. This ranking allows them to reward high performers and support those falling behind.</span>

In [15]:
Use AdventureWorks2019
GO

WITH SalesByRep AS (
    SELECT 
        soh.SalesPersonID,
        st.Name AS Territory,
        SUM(soh.TotalDue) AS TotalSales
    FROM Sales.SalesOrderHeader soh
    JOIN Sales.SalesTerritory st ON soh.TerritoryID = st.TerritoryID
    WHERE soh.SalesPersonID IS NOT NULL
    GROUP BY soh.SalesPersonID, st.Name
),
RankedSales AS (
    SELECT *,
           RANK() OVER (PARTITION BY Territory ORDER BY TotalSales DESC) AS RankInTerritory
    FROM SalesByRep
)
SELECT *
FROM RankedSales
WHERE RankInTerritory <= 3;


Commands completed successfully.

(26 rows affected)

Total execution time: 00:00:00.035

SalesPersonID,Territory,TotalSales,RankInTerritory
286,Australia,1606441.4471,1
285,Australia,195528.7838,2
289,Canada,9585124.9477,1
278,Canada,4069422.2109,2
282,Canada,2354403.7626,3
277,Central,3311501.8521,1
275,Central,2899017.4537,2
276,Central,2077158.6286,3
290,France,5087977.212,1
287,France,110132.4925,2


### **2\. Proposition: Total Sales Amount by Product Subcategory, Category, and Grand Total**

Marketing and product teams need insights into how much revenue is generated by different categories and subcategories to guide promotions, inventory, and product development strategies.

In [16]:
Use AdventureWorks2019
GO

SELECT 
    pc.Name AS Category,
    psc.Name AS Subcategory,
    SUM(sod.LineTotal) AS TotalRevenue
FROM Sales.SalesOrderDetail sod
JOIN Production.Product p ON sod.ProductID = p.ProductID
JOIN Production.ProductSubcategory psc ON p.ProductSubcategoryID = psc.ProductSubcategoryID
JOIN Production.ProductCategory pc ON psc.ProductCategoryID = pc.ProductCategoryID
GROUP BY ROLLUP(pc.Name, psc.Name);


Commands completed successfully.

(40 rows affected)

Total execution time: 00:00:00.117

Category,Subcategory,TotalRevenue
Accessories,Bike Racks,237096.156000
Accessories,Bike Stands,39591.000000
Accessories,Bottles and Cages,64274.793327
Accessories,Cleaners,18406.972080
Accessories,Fenders,46619.580000
Accessories,Helmets,484048.529277
Accessories,Hydration Packs,105826.418334
Accessories,Locks,16240.220000
Accessories,Pumps,13514.687276
Accessories,Tires and Tubes,246454.527632


**3\. Proposition: Compare Each Customer’s Order Value to Their Previous and Next Order**

**  
Customer behavior analysts want to track purchasing trends at the individual level. This helps identify decreasing or increasing spending patterns, which could indicate loyalty, churn risk, or opportunities for upselling.  
  
**

In [17]:
Use AdventureWorks2019
GO

WITH CustomerOrders AS (
    SELECT 
        CustomerID,
        SalesOrderID,
        OrderDate,
        TotalDue,
        LAG(TotalDue) OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS PrevOrderTotal,
        LEAD(TotalDue) OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS NextOrderTotal
    FROM Sales.SalesOrderHeader
)
SELECT *
FROM CustomerOrders
WHERE PrevOrderTotal IS NOT NULL OR NextOrderTotal IS NOT NULL;


Commands completed successfully.

(19816 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.139

CustomerID,SalesOrderID,OrderDate,TotalDue,PrevOrderTotal,NextOrderTotal
11000,43793,2011-06-21 00:00:00.000,3756.989,NULL,2587.8769
11000,51522,2013-06-20 00:00:00.000,2587.8769,3756.989,2770.2682
11000,57418,2013-10-03 00:00:00.000,2770.2682,2587.8769,NULL
11001,43767,2011-06-17 00:00:00.000,3729.364,NULL,2674.0227
11001,51493,2013-06-18 00:00:00.000,2674.0227,3729.364,650.8008
11001,72773,2014-05-12 00:00:00.000,650.8008,2674.0227,NULL
11002,43736,2011-06-09 00:00:00.000,3756.989,NULL,2535.964
11002,51238,2013-06-02 00:00:00.000,2535.964,3756.989,2673.0613
11002,53237,2013-07-26 00:00:00.000,2673.0613,2535.964,NULL
11003,43701,2011-05-31 00:00:00.000,3756.989,NULL,2562.4508


### **4) Proposition: Monthly Sales Pivot by Territory for the Year**

Executives want to view monthly sales performance for each region to assess market seasonality and territory trends. This helps with regional planning and staffing decisions.

In [18]:
Use AdventureWorks2019
GO

SELECT *
FROM (
    SELECT 
        st.Name AS Territory,
        MONTH(soh.OrderDate) AS OrderMonth,
        soh.TotalDue
    FROM Sales.SalesOrderHeader soh
    JOIN Sales.SalesTerritory st ON soh.TerritoryID = st.TerritoryID
    WHERE YEAR(soh.OrderDate) = 2013
) AS SourceData
PIVOT (
    SUM(TotalDue) FOR OrderMonth IN ([1], [2], [3], [4], [5], [6],
                                     [7], [8], [9], [10], [11], [12])
) AS PivotTable
ORDER BY Territory;


Commands completed successfully.

(10 rows affected)

Total execution time: 00:00:00.084

Territory,1,2,3,4,5,6,7,8,9,10,11,12
Australia,245877.6662,196345.3108,238335.0813,226708.5813,281113.4128,425665.2739,513157.5061,437404.2109,507732.4517,636814.2598,473749.5875,519500.7081
Canada,469186.4587,519516.2921,546868.6262,525728.9935,576366.9462,685595.7924,830057.7188,525099.5787,540507.2667,904192.4585,398709.4663,488620.1013
Central,202261.2412,218487.6571,319856.0998,248029.4033,244261.5401,427596.4269,392829.8676,182924.001,365911.419,357515.0421,128235.4309,286428.1702
France,121103.7535,137170.7643,276427.5385,154563.5524,168549.7237,906054.8621,315764.7213,182206.0084,802984.3449,302580.9188,249111.1278,654501.9506
Germany,76864.5552,62590.9246,73427.2432,62339.4351,285165.5161,408022.7238,253737.3584,345174.1461,329338.7763,309418.6472,319691.3244,343721.3208
Northeast,171608.4392,377471.5013,384092.3222,201615.3594,215034.07,323580.4705,309096.1495,212906.1634,204013.4618,250009.7312,157785.4202,158353.9397
Northwest,132466.2661,291095.5501,613562.8536,166660.6948,487428.2923,925474.2621,794164.6992,469177.0781,799611.4305,758382.8113,607461.2987,714015.4345
Southeast,157417.6261,196642.7644,240114.7286,235435.8362,200163.8831,310417.2471,328279.6543,181786.2846,263184.3583,247661.2059,129411.7493,215215.6316
Southwest,556150.6565,485072.5154,952570.7864,798483.3107,960385.9368,929320.2688,1088582.5182,914450.2527,881995.0677,933249.3282,900406.9175,838541.7814
United Kingdom,207124.8894,115825.5866,186350.6591,221146.0067,239615.625,384537.9359,696170.6511,282845.2793,388226.7605,674551.5388,330105.6772,341678.0576


### **5\. Proposition: Average Line Total per Product for Products in More Than 10 Orders**

**Business Use Case:**  
Product managers want to focus on items that sell consistently, filtering out one-off purchases. This helps target reliable, high-performing products for forecasting and stocking.

In [8]:
Use AdventureWorks2019
GO

SELECT 
    p.Name AS ProductName,
    COUNT(*) AS OrderCount,
    AVG(sod.LineTotal) AS AverageLineTotal
FROM Sales.SalesOrderDetail sod
JOIN Production.Product p ON sod.ProductID = p.ProductID
GROUP BY p.Name
HAVING COUNT(*) > 10;


Commands completed successfully.

(260 rows affected)

Total execution time: 00:00:00.065

ProductName,OrderCount,AverageLineTotal
All-Purpose Bike Stand,249,159.000000
AWC Logo Cap,3382,15.147677
Bike Wash - Dissolver,1327,13.871116
Cable Lock,260,62.462384
Chain,250,37.510840
"Classic Vest, L",201,63.879104
"Classic Vest, M",555,162.613694
"Classic Vest, S",682,229.322680
Fender Set - Mountain,2121,21.980000
Front Brakes,266,189.095154


### 6\. Proposition: Orders with Above-Average Freight Cost for Their Territory

**Business Use Case:**  
Logistics managers monitor freight spending per region to identify abnormal shipping costs. This can expose inefficiencies, fraud, or the need to renegotiate shipping rates.

In [19]:
Use AdventureWorks2019
GO

WITH FreightAnalysis AS (
    SELECT 
        soh.SalesOrderID,
        soh.TerritoryID,
        soh.Freight,
        AVG(soh.Freight) OVER (PARTITION BY TerritoryID) AS AvgFreight
    FROM Sales.SalesOrderHeader soh
    WHERE soh.TerritoryID IS NOT NULL
)
SELECT *
FROM FreightAnalysis
WHERE Freight > AvgFreight;


Commands completed successfully.

(6287 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.086

SalesOrderID,TerritoryID,Freight,AvgFreight
43664,1,732.81,102.4482
43665,1,429.9821,102.4482
43671,1,244.0042,102.4482
43683,1,1283.484,102.4482
43686,1,103.899,102.4482
43848,1,548.0984,102.4482
43849,1,615.6177,102.4482
43860,1,330.4013,102.4482
43866,1,122.2584,102.4482
43867,1,984.1535,102.4482


### **7\. Proposition: Grouping Sets for Total Sales by Territory and Year**

**Business Use Case:**  
Finance teams want a compact summary of total sales broken down by different combinations of territory and order year. Grouping sets allow them to see this all in one query instead of writing multiple ones.

In [21]:
Use AdventureWorks2019
GO

SELECT 
    st.Name AS Territory,
    YEAR(soh.OrderDate) AS OrderYear,
    SUM(soh.TotalDue) AS TotalSales,
    GROUPING_ID(st.Name, YEAR(soh.OrderDate)) AS GroupingID
FROM Sales.SalesOrderHeader soh
JOIN Sales.SalesTerritory st ON soh.TerritoryID = st.TerritoryID
GROUP BY GROUPING SETS (
    (st.Name, YEAR(soh.OrderDate)),
    (st.Name),
    (YEAR(soh.OrderDate)),
    ()
)
ORDER BY GroupingID;


Commands completed successfully.

(55 rows affected)

Total execution time: 00:00:00.020

Territory,OrderYear,TotalSales,GroupingID
Australia,2011,1693032.7418,0
Canada,2011,2106905.8728,0
Central,2011,1126645.7497,0
France,2011,236268.627,0
Germany,2011,272780.9066,0
Northeast,2011,705672.1952,0
Northwest,2011,2620943.826,0
Southeast,2011,1847744.578,0
Southwest,2011,3144713.0989,0
United Kingdom,2011,400991.929,0


### 8\. Proposition: Rank Product Categories by Total Revenue

**Business Use Case:**  
Executives need to rank product categories to focus attention and funding on the most profitable areas. This ranking supports strategic decision-making and product lifecycle planning.

In [22]:
Use AdventureWorks2019
GO

WITH CategorySales AS (
    SELECT 
        pc.Name AS Category,
        SUM(sod.LineTotal) AS Revenue
    FROM Sales.SalesOrderDetail sod
    JOIN Production.Product p ON sod.ProductID = p.ProductID
    JOIN Production.ProductSubcategory psc ON p.ProductSubcategoryID = psc.ProductSubcategoryID
    JOIN Production.ProductCategory pc ON psc.ProductCategoryID = pc.ProductCategoryID
    GROUP BY pc.Name
)
SELECT 
    Category,
    Revenue,
    DENSE_RANK() OVER (ORDER BY Revenue DESC) AS RevenueRank
FROM CategorySales;


Commands completed successfully.

(4 rows affected)

Total execution time: 00:00:00.133

Category,Revenue,RevenueRank
Bikes,94651172.704731,1
Components,11802593.286430,2
Clothing,2120542.524801,3
Accessories,1272072.883926,4


### **9\. Proposition: Customers Who Ordered All of the Top 3 Selling Products**

**Business Use Case:**  
Marketing wants to identify highly engaged customers who have purchased the most popular products. These customers are ideal for upselling, loyalty programs, and case studies.

In [23]:
Use AdventureWorks2019
GO

WITH TopProducts AS (
    SELECT TOP 3 ProductID
    FROM Sales.SalesOrderDetail
    GROUP BY ProductID
    ORDER BY SUM(LineTotal) DESC
),
CustomerProducts AS (
    SELECT soh.CustomerID, sod.ProductID
    FROM Sales.SalesOrderHeader soh
    JOIN Sales.SalesOrderDetail sod ON soh.SalesOrderID = sod.SalesOrderID
    WHERE sod.ProductID IN (SELECT ProductID FROM TopProducts)
)
SELECT CustomerID
FROM CustomerProducts
GROUP BY CustomerID
HAVING COUNT(DISTINCT ProductID) = 3;


Commands completed successfully.

(102 rows affected)

Total execution time: 00:00:00.091

CustomerID
29484
29492
29494
29497
29499
29500
29507
29521
29522
29523


### **10\. Proposition: Identify Customers Who Made Repeat Purchases Within 30 Days**

**Business Use Case:**  
The marketing department wants to identify repeat customers who place orders within a short time span (e.g., within 30 days). These customers may be more loyal or receptive to targeted promotions and reward programs.

In [24]:
Use AdventureWorks2019
GO

WITH CustomerOrders AS (
    SELECT 
        CustomerID,
        SalesOrderID,
        OrderDate,
        LAG(OrderDate) OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS PrevOrderDate
    FROM Sales.SalesOrderHeader
)
SELECT 
    CustomerID,
    SalesOrderID,
    OrderDate,
    PrevOrderDate,
    DATEDIFF(DAY, PrevOrderDate, OrderDate) AS DaysBetween
FROM CustomerOrders
WHERE PrevOrderDate IS NOT NULL AND DATEDIFF(DAY, PrevOrderDate, OrderDate) <= 30;


Commands completed successfully.

(1264 rows affected)

Total execution time: 00:00:00.071

CustomerID,SalesOrderID,OrderDate,PrevOrderDate,DaysBetween
11019,53834,2013-08-04 00:00:00.000,2013-07-15 00:00:00.000,20
11019,54332,2013-08-13 00:00:00.000,2013-08-04 00:00:00.000,9
11019,57639,2013-10-08 00:00:00.000,2013-09-28 00:00:00.000,10
11019,58600,2013-10-25 00:00:00.000,2013-10-08 00:00:00.000,17
11019,64737,2014-01-22 00:00:00.000,2013-12-23 00:00:00.000,30
11019,65967,2014-02-08 00:00:00.000,2014-01-22 00:00:00.000,17
11019,66658,2014-02-19 00:00:00.000,2014-02-08 00:00:00.000,11
11019,68563,2014-03-19 00:00:00.000,2014-02-19 00:00:00.000,28
11019,69177,2014-03-28 00:00:00.000,2014-03-19 00:00:00.000,9
11019,72015,2014-05-02 00:00:00.000,2014-05-01 00:00:00.000,1
